[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/23_probabilistic_machine_learning/01_multivariate_gaussian.ipynb)

# Bài 01 — Phân phối Gaussian nhiều chiều

**🎯 Mục tiêu**
- Mật độ Gaussian nhiều chiều; **hình học** của ma trận hiệp phương sai (trị riêng/vector riêng ↔ ellipse).
- Lấy mẫu bằng **Cholesky**; biến đổi tuyến tính của Gaussian.
- **Phân phối biên và có điều kiện** — công thức nền tảng của Gaussian Process (bài 04) và Kalman filter; kỳ vọng có điều kiện = đường hồi quy tuyến tính.
- Ước lượng MLE và vấn đề khi $d$ lớn so với $n$ (shrinkage); bộ phân loại sinh: **LDA, QDA** và quan hệ với Gaussian Naive Bayes.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/23_probabilistic_machine_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import multivariate_normal
from sklearn.covariance import LedoitWolf
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import GaussianNB

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Mật độ và hình dạng

$$\mathcal N(\mathbf x \mid \boldsymbol\mu, \Sigma) = \frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}}\exp\left(-\tfrac12(\mathbf x - \boldsymbol\mu)^\top\Sigma^{-1}(\mathbf x - \boldsymbol\mu)\right)$$

- $\boldsymbol\mu$: tâm; $\Sigma$ (đối xứng, xác định dương): hình dạng. Đường đồng mức là các **ellipse** $(\mathbf x - \boldsymbol\mu)^\top\Sigma^{-1}(\mathbf x - \boldsymbol\mu) = c$ (khoảng cách Mahalanobis không đổi — chương 22).
- $|\Sigma|$ ("thể tích" của đám mây) nằm ở mẫu số để mật độ tích phân bằng 1.

In [ ]:
covs = {"đẳng hướng σ²I": [[1, 0], [0, 1]], "chéo (độc lập)": [[2.5, 0], [0, 0.4]],
        "tương quan dương 0.8": [[1, 0.8], [0.8, 1]], "tương quan âm ≈ −0.78": [[2, -0.85], [-0.85, 0.6]]}
g = np.linspace(-4, 4, 200)
GX, GY = np.meshgrid(g, g)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, S) in zip(axes, covs.items()):
    S = np.array(S, float)
    ax.contour(GX, GY, multivariate_normal([0, 0], S).pdf(np.dstack([GX, GY])), levels=6, cmap="viridis")
    ax.scatter(*rng.multivariate_normal([0, 0], S, 300).T, s=4, alpha=0.4)
    vals, vecs = np.linalg.eigh(S)
    for lam, v in zip(vals, vecs.T):                                  # trục ellipse: vector riêng, dài ∝ √λ
        ax.arrow(0, 0, *(2 * np.sqrt(lam) * v), width=0.04, color="r")
    ax.set_title(f"{name}\nλ = {vals.round(2)}")
    ax.set_aspect("equal")
plt.show()

**Hình học:** phân rã $\Sigma = Q\Lambda Q^\top$. Các trục ellipse nằm theo **vector riêng** (cột của $Q$), bán trục tỉ lệ với $\sqrt{\lambda_j}$ (mũi tên đỏ dài $2\sqrt\lambda$). Đổi sang hệ trục vector riêng, các thành phần **không tương quan** — chính là PCA (chương 21).

## 2. Lấy mẫu bằng Cholesky và biến đổi tuyến tính

Nếu $\mathbf z \sim \mathcal N(\mathbf 0, I)$ và $\mathbf x = A\mathbf z + \mathbf b$ thì $\mathbf x \sim \mathcal N(\mathbf b, AA^\top)$. Muốn hiệp phương sai $\Sigma$ → chọn $A = L$ với $\Sigma = LL^\top$ (**Cholesky**):

In [ ]:
mu = np.array([1.0, -2.0, 0.5])
Sigma = np.array([[2.0, 0.8, -0.5], [0.8, 1.0, 0.3], [-0.5, 0.3, 1.5]])
L = np.linalg.cholesky(Sigma)
Z = rng.standard_normal((100_000, 3))
X = mu + Z @ L.T
print("L =\n", L.round(3))
print("trung bình mẫu:", X.mean(0).round(3), "| hiệp phương sai mẫu − Σ: sai lệch lớn nhất =", np.abs(np.cov(X.T) - Sigma).max().round(4))

A = np.array([[1, 1, 0], [0, 1, -1]])                                  # biến đổi tuyến tính 3D → 2D
Y = X @ A.T
print("Cov(AX) mẫu:\n", np.cov(Y.T).round(3), "\nlý thuyết AΣAᵀ:\n", (A @ Sigma @ A.T).round(3))

Hệ quả: **tổng** các biến Gaussian (kể cả tương quan) vẫn là Gaussian — $\text{Var}(x_1 + x_2) = \Sigma_{11} + \Sigma_{22} + 2\Sigma_{12}$. Cholesky cũng là cách tính ổn định cho $\log|\Sigma| = 2\sum_i\log L_{ii}$ và giải $\Sigma^{-1}\mathbf b$ mà không nghịch đảo ma trận (dùng ở bài 04).

## 3. Phân phối biên và có điều kiện

Chia $\mathbf x = \begin{bmatrix}\mathbf x_a \\ \mathbf x_b\end{bmatrix}$, $\boldsymbol\mu = \begin{bmatrix}\boldsymbol\mu_a \\ \boldsymbol\mu_b\end{bmatrix}$, $\Sigma = \begin{bmatrix}\Sigma_{aa} & \Sigma_{ab} \\ \Sigma_{ba} & \Sigma_{bb}\end{bmatrix}$.

- **Biên**: $p(\mathbf x_a) = \mathcal N(\boldsymbol\mu_a, \Sigma_{aa})$ — chỉ việc "bỏ" các thành phần còn lại.
- **Có điều kiện**: $p(\mathbf x_a \mid \mathbf x_b) = \mathcal N(\boldsymbol\mu_{a|b}, \Sigma_{a|b})$ với
$$\boldsymbol\mu_{a|b} = \boldsymbol\mu_a + \Sigma_{ab}\Sigma_{bb}^{-1}(\mathbf x_b - \boldsymbol\mu_b), \qquad \Sigma_{a|b} = \Sigma_{aa} - \Sigma_{ab}\Sigma_{bb}^{-1}\Sigma_{ba}$$

Ví dụ: chiều cao (cm) và cân nặng (kg) của người trưởng thành, tương quan 0.6. Biết cân nặng 75 kg, dự đoán chiều cao:

In [ ]:
mu_hw = np.array([165.0, 60.0])                                         # [chiều cao, cân nặng]
sd_h, sd_w, rho = 8.0, 10.0, 0.6
S_hw = np.array([[sd_h**2, rho * sd_h * sd_w], [rho * sd_h * sd_w, sd_w**2]])


def condition(mu, S, idx_a, idx_b, x_b):
    S_ab, S_bb = S[np.ix_(idx_a, idx_b)], S[np.ix_(idx_b, idx_b)]
    K = S_ab @ np.linalg.inv(S_bb)
    return mu[idx_a] + K @ (x_b - mu[idx_b]), S[np.ix_(idx_a, idx_a)] - K @ S_ab.T


m_cond, S_cond = condition(mu_hw, S_hw, [0], [1], np.array([75.0]))
people = rng.multivariate_normal(mu_hw, S_hw, 500_000)
near = people[np.abs(people[:, 1] - 75) < 0.5, 0]                       # Monte Carlo: những người nặng ≈ 75 kg
print(f"công thức: chiều cao | 75 kg ~ N({m_cond[0]:.2f}, {np.sqrt(S_cond[0, 0]):.2f}²)")
print(f"Monte Carlo ({len(near):,} người nặng 74.5–75.5 kg): trung bình {near.mean():.2f}, độ lệch chuẩn {near.std():.2f}")
print(f"không biết cân nặng: N(165, 8²) → biết cân nặng giảm độ bất định {1 - np.sqrt(S_cond[0, 0]) / sd_h:.0%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(*people[:3000, ::-1].T, s=3, alpha=0.3)
ws = np.linspace(30, 90, 50)
axes[0].plot(ws, [condition(mu_hw, S_hw, [0], [1], np.array([w]))[0][0] for w in ws], "r", lw=2, label="E[chiều cao | cân nặng]")
axes[0].axvline(75, color="k", ls="--")
axes[0].set_xlabel("cân nặng (kg)")
axes[0].set_ylabel("chiều cao (cm)")
axes[0].legend()
hs = np.linspace(135, 195, 300)
axes[1].plot(hs, multivariate_normal(165, 64).pdf(hs), label="biên p(chiều cao)")
axes[1].plot(hs, multivariate_normal(m_cond[0], S_cond[0, 0]).pdf(hs), label="có điều kiện p(chiều cao | 75 kg)")
axes[1].hist(near, bins=40, density=True, alpha=0.3, label="Monte Carlo")
axes[1].legend()
plt.show()

Ba nhận xét quan trọng:
1. Kỳ vọng có điều kiện **tuyến tính** theo $\mathbf x_b$ — đường đỏ chính là **đường hồi quy tuyến tính** của chiều cao theo cân nặng (độ dốc $\rho\,\sigma_h/\sigma_w$). Hồi quy tuyến tính là tối ưu khi dữ liệu đúng là Gaussian chung.
2. Phương sai có điều kiện **không phụ thuộc** giá trị $\mathbf x_b$ quan sát được, luôn ≤ phương sai biên: quan sát thêm không bao giờ làm tăng bất định (với Gaussian).
3. Bài 04: Gaussian Process chính là áp dụng công thức này với $\mathbf x_a$ = giá trị hàm tại điểm cần dự đoán, $\mathbf x_b$ = giá trị đã quan sát.

## 4. Ước lượng MLE và khi $d$ lớn

MLE: $\hat{\boldsymbol\mu} = \bar{\mathbf x}$, $\hat\Sigma = \frac1n\sum_i(\mathbf x_i - \bar{\mathbf x})(\mathbf x_i - \bar{\mathbf x})^\top$ — có $d + d(d+1)/2$ tham số. Khi $n$ không lớn hơn nhiều so với $d$, $\hat\Sigma$ **méo** (trị riêng lớn bị phóng đại, nhỏ bị co lại), thậm chí **suy biến** khi $n \le d$. **Shrinkage** (Ledoit–Wolf) kéo $\hat\Sigma$ về một ma trận đơn giản: $(1 - \delta)\hat\Sigma + \delta\,\frac{\text{tr}\hat\Sigma}{d}I$.

In [ ]:
d = 40
true_S = 0.6 ** np.abs(np.subtract.outer(np.arange(d), np.arange(d)))  # tương quan giảm dần theo khoảng cách
X_test = rng.multivariate_normal(np.zeros(d), true_S, 5000)
print(f"{'n':>5} | {'rank Σ̂':>6} | log-lik test/điểm: MLE | Ledoit-Wolf | Σ thật")
for n in [30, 60, 200, 2000]:
    Xn = rng.multivariate_normal(np.zeros(d), true_S, n)
    S_mle = np.cov(Xn.T, bias=True)
    lw = LedoitWolf().fit(Xn)
    ll = lambda S, m: multivariate_normal(m, S, allow_singular=True).logpdf(X_test).mean()
    mle_ll = ll(S_mle, Xn.mean(0)) if np.linalg.matrix_rank(S_mle) == d else -np.inf
    print(f"{n:5d} | {np.linalg.matrix_rank(S_mle):6d} | {mle_ll:22.2f} | {ll(lw.covariance_, lw.location_):11.2f} | {ll(true_S, np.zeros(d)):6.2f}")

Với $n = 30 < d = 40$, MLE suy biến (không có mật độ — log-likelihood $-\infty$); với $n = 60$ vẫn tệ hơn hẳn shrinkage. Khi $n$ lớn, hai cách hội tụ về nhau. Đây cũng là lý do Gaussian Naive Bayes (chỉ dùng đường chéo, chương 09) hay thắng khi ít dữ liệu.

## 5. Bộ phân loại sinh: LDA và QDA

Mô hình hoá mỗi lớp bằng một Gaussian $p(\mathbf x \mid y = k) = \mathcal N(\boldsymbol\mu_k, \Sigma_k)$, rồi dùng Bayes: $p(y = k \mid \mathbf x) \propto \pi_k\,\mathcal N(\mathbf x \mid \boldsymbol\mu_k, \Sigma_k)$.

| | Hiệp phương sai | Ranh giới | Tham số |
|---|---|---|---|
| Gaussian NB | mỗi lớp, **chéo** | bậc hai (song song trục) | $2Kd$ |
| **LDA** | **chung** cho mọi lớp, đầy đủ | **tuyến tính** (số hạng bậc hai triệt tiêu) | $Kd + d(d+1)/2$ |
| **QDA** | **riêng** mỗi lớp, đầy đủ | bậc hai | $K(d + d(d+1)/2)$ |

In [ ]:
def two_classes(n, seed):
    r = np.random.default_rng(seed)
    X0 = r.multivariate_normal([0, 0], [[2, 1.7], [1.7, 2]], n)                 # tương quan +0.85
    X1 = r.multivariate_normal([0.8, -0.8], [[1.2, -1.02], [-1.02, 1.2]], n)     # tương quan −0.85
    return np.vstack([X0, X1]), np.r_[np.zeros(n), np.ones(n)]


Xc, yc = two_classes(300, seed=1)
models = {"Gaussian NB": GaussianNB(), "LDA": LinearDiscriminantAnalysis(), "QDA": QuadraticDiscriminantAnalysis()}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
gx, gy = np.meshgrid(np.linspace(-5, 6, 300), np.linspace(-6, 5, 300))
for ax, (name, m) in zip(axes, models.items()):
    acc = cross_val_score(m, Xc, yc, cv=5).mean()
    m.fit(Xc, yc)
    ax.contourf(gx, gy, m.predict_proba(np.c_[gx.ravel(), gy.ravel()])[:, 1].reshape(gx.shape), levels=20, cmap="RdBu_r", alpha=0.4)
    ax.scatter(*Xc.T, c=yc, cmap="RdBu_r", s=8, edgecolors="k", linewidths=0.2)
    ax.set_title(f"{name}: accuracy CV = {acc:.3f}")
    print(f"{name:12s} accuracy CV = {acc:.3f}")
plt.show()

Hai lớp có hiệp phương sai **khác nhau** (tương quan ngược dấu, hình chữ X): LDA buộc dùng chung một $\Sigma$ → ranh giới thẳng; Gaussian NB bỏ qua tương quan — cả hai kém QDA rõ rệt, vì chỉ QDA khớp đúng giả định (mỗi lớp một ellipse riêng). Ngược lại, khi ít dữ liệu và nhiều chiều, LDA (ít tham số) thường thắng QDA — đánh đổi bias–variance quen thuộc. LDA còn là một phương pháp **giảm chiều có giám sát** (chiếu lên $K - 1$ chiều phân biệt lớp tốt nhất).

## 🧠 Kiểm tra nhanh

1. Trục của ellipse đồng mức liên hệ thế nào với $\Sigma$?
2. Lấy mẫu từ $\mathcal N(\boldsymbol\mu, \Sigma)$ chỉ với bộ sinh $\mathcal N(0, 1)$ thế nào?
3. Viết $\boldsymbol\mu_{a|b}$, $\Sigma_{a|b}$. Vì sao $\Sigma_{a|b}$ không phụ thuộc $\mathbf x_b$?
4. Vì sao ước lượng $\Sigma$ bằng MLE thất bại khi $n \le d$? Cách khắc phục?
5. LDA và QDA khác nhau ở giả định nào? Hệ quả với ranh giới quyết định?

## 📌 Tóm tắt

- $\Sigma = Q\Lambda Q^\top$: ellipse theo vector riêng, bán trục $\propto\sqrt\lambda$; lấy mẫu $\boldsymbol\mu + L\mathbf z$ (Cholesky).
- Biên: bỏ thành phần; có điều kiện: $\boldsymbol\mu_a + \Sigma_{ab}\Sigma_{bb}^{-1}(\mathbf x_b - \boldsymbol\mu_b)$ — tuyến tính, phương sai giảm.
- $d$ lớn so với $n$: shrinkage; bộ phân loại sinh: NB (chéo) → LDA (chung) → QDA (riêng).

## 📚 Tài liệu tham khảo

- Bishop (2006) — *Pattern Recognition and Machine Learning*, §2.3
- Murphy (2022) — *Probabilistic Machine Learning: An Introduction*, ch. 3 & 9
- Ledoit & Wolf (2004) — *A well-conditioned estimator for large-dimensional covariance matrices*